# Chapter 33 — Shock, Transient, and Fatigue Measurement
### *Modern Strain Gage Technology* — Companion Notebook

**A cracking strain gage does not fail quietly — it lies, and it lies in a way you can watch on an oscilloscope.**

Companion notebook to Chapter 33 — Shock, Transient, and Fatigue Measurement.

This notebook builds the *super-sensitivity* figure that accompanies the gage-fatigue discussion. *Super-sensitivity* is the oscilloscope signature of a forming fatigue crack in the gage grid: the crack opens only during the tension half of each cycle, adding extra resistance change while it is open, so the gage reads anomalously **high in tension** while the compression half stays clean. We model that one-sided distortion, then show why it sprouts **even** harmonics and a small **DC** shift — a quantitative fingerprint to go with the visual one.

The notebook is short by design: one imports-and-configuration cell, then a single cell that builds the model and renders the figure.

In [ ]:
# !pip install numpy matplotlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# The book stores this chapter's figures here; we save with the exact filename the .tex expects.
OUT_DIR = Path('../src/media/ch33')
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

## Super-Sensitivity: Tension-Only Distortion

A partial crack across the grid reduces the conducting cross-section. In **tension** the crack opens and the resistance jumps above the normal strain response, so the indicated strain overshoots; in **compression** the crack closes and the gage behaves normally. The result is a waveform that departs from the clean sine on the tension half only. Because the distortion is one-sided, it breaks the half-wave symmetry of the signal and introduces **even** harmonics (2f, 4f, ...) and a small **DC** shift — a quantitative fingerprint to go with the visual one.

In [ ]:
# Model parameters. The signal is a deterministic synthetic sine, so there is
# nothing random to seed; the spectrum is coherent (integer cycles per record).
f0 = 10.0          # fatigue cycling frequency (Hz)
fs = 2000.0        # sample rate for the model (Hz)
N  = int(fs)       # 1 s -> 10 whole cycles, coherent (clean spectrum)
t  = np.arange(N) / fs
eps0 = 1500.0      # tension/compression strain amplitude (microstrain)
base = eps0 * np.sin(2 * np.pi * f0 * t)

# Fatigue crack opens only in tension, above a threshold; while open it adds
# extra resistance change, so the gage reads anomalously HIGH in tension.
eps_open   = 0.30 * eps0   # crack begins to open at this tension level (microstrain)
crack_gain = 0.7           # extra sensitivity once the crack is open (dimensionless)
extra   = crack_gain * np.clip(base - eps_open, 0, None)
super_s = base + extra


def spec(y):
    """Single-sided amplitude spectrum of ``y``, normalized to the fundamental.

    Returns the frequency axis (Hz) and the amplitude at each bin scaled so the
    component nearest ``f0`` equals 1.0. Real-FFT amplitudes are doubled to fold
    the negative-frequency energy back in, except the DC term, which is unique.
    """
    Y = np.abs(np.fft.rfft(y)) * 2.0 / N
    Y[0] /= 2.0                       # DC term is not doubled
    f = np.fft.rfftfreq(N, 1.0 / fs)
    i0 = np.argmin(np.abs(f - f0))
    return f, Y / Y[i0]               # normalize to the fundamental

f_b, Y_b = spec(base)
f_s, Y_s = spec(super_s)

fig, (axW, axS) = plt.subplots(1, 2, figsize=(13, 5.0))

# ---- (a) oscilloscope view, 2 cycles ----
tms = t * 1000.0
m = tms <= 2000.0 / f0             # 2 cycles, in ms (= 200 ms)
axW.set_ylim(-1900, 2700)         # headroom so annotations stay inside the axes
axW.axhline(0, color='0.7', lw=0.8)
axW.plot(tms[m], base[m], color='0.5', ls='--', lw=1.6, label='undamaged gage (clean sine)')
axW.plot(tms[m], super_s[m], color='#c0392b', lw=2.2, label='cracking gage (super-sensitivity)')
axW.fill_between(tms[m], 0, super_s[m], where=(super_s[m] > 0), color='tomato', alpha=0.10)
axW.axhline(eps_open, color='purple', ls=':', lw=1.2)
axW.text(3, eps_open + 50, 'crack opens (tension only)', fontsize=8, color='purple')
pk = int(np.argmax(super_s[m]))
axW.annotate('apparent strain reads high\n(super-sensitivity)',
             xy=(tms[m][pk], super_s[m][pk]),
             xytext=(tms[m][pk] + 24, super_s[m][pk] - 250),
             fontsize=8, color='#c0392b',
             arrowprops=dict(arrowstyle='->', color='#c0392b'))
axW.text(75, -1500, 'compression half:\ncrack closed, trace clean',
         ha='center', va='top', fontsize=8, color='0.35')
axW.set_xlabel('Time (ms)')
axW.set_ylabel(r'Indicated strain  ($\mu\varepsilon$)')
axW.set_title('(a) Oscilloscope view: tension-only distortion')
axW.legend(loc='upper right', fontsize=8)
axW.grid(True, ls=':', alpha=0.4)

# ---- (b) spectrum ----
fmax = 85
sel = f_s <= fmax
axS.stem(f_s[sel], Y_s[sel], linefmt='#c0392b', markerfmt=' ', basefmt=' ')
axS.annotate('DC', xy=(0, Y_s[0]), xytext=(2, Y_s[0] + 0.04), fontsize=8, color='purple')
for n in (2, 3, 4, 5, 6):
    fh = n * f0
    amp = Y_s[np.argmin(np.abs(f_s - fh))]
    if amp > 0.01:
        col = 'green' if n % 2 == 0 else '#c0392b'
        axS.annotate(f'{n}f', xy=(fh, amp), xytext=(fh, amp + 0.04),
                     ha='center', fontsize=8, color=col)
axS.set_ylim(0, 1.1)
axS.set_xlabel('Frequency (Hz)')
axS.set_ylabel('Amplitude (rel. to fundamental)')
axS.set_title('(b) Spectrum: even harmonics + DC (even in green)')
axS.grid(True, ls=':', alpha=0.4)

fig.suptitle('Super-Sensitivity: a Fatigue Crack Distorts Only the Tension Half-Cycle',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig33_nb1_super_sensitivity.png', bbox_inches='tight', dpi=150)
plt.show()

## Where to take this next

- **Sweep the crack severity.** Turn `crack_gain` and `eps_open` into a small grid and plot the second-harmonic amplitude (2*f*) against crack gain. That curve is the start of a quantitative *crack-detection metric* you could threshold in software, instead of eyeballing the trace.
- **Add measurement noise and detect the tell automatically.** Superimpose band-limited noise on `super_s`, then write a detector that flags a channel when the 2*f*/1*f* ratio or the DC-to-fundamental ratio crosses a set limit — the automated version of the experimenter watching the oscilloscope, and a first step toward the machine-learning methods in Part 9.
- **Model both gage-factor tells at once.** The chapter notes that cracking drives the *tension* gage factor up while the *compression* value lags. Give tension and compression separate sensitivities (not just a one-sided tension term) and compare the resulting waveform and spectrum with the single-sided model here.